# Step 04. 최적화 기반 궤적 계획 (Optimization-based Trajectory Planning)

> **교육 주관**: Hanyang University, Automotive Intelligence Laboratory (AILAB)

Step 03에서는 미리 만든 여러 후보 경로 중 비용이 가장 낮은 것을 **골랐다면**, 이 노트북에서는 후보를 만들지 않고
조향 계획(control input) 자체를 solver가 **직접 최적화**합니다. 비용 함수(soft)와 안전 제약(hard)으로
"좋은 궤적"을 정의하면, solver가 그 기준에서 가장 좋은 궤적을 찾아 줍니다.

**미션**: 파란 차가 **초록 reference path**를 따라가고 **빨간 차**를 피하게 만들어라!


## 학습 목표

1. Trajectory planning을 **최적화 문제**(결정변수 · 비용 · 제약)로 정식화하는 관점 이해
2. Kinematic bicycle model — 조향 계획을 미래 궤적으로 펼치기 (TODO 1)
3. 결정변수 $z$(조향 계획)를 직접 만들어 보기 (TODO 2, 직접 해보기 ①)
4. Cost function(tracking · heading · comfort) 구현과 내 계획 채점 (TODO 3~7, 직접 해보기 ②)
5. Hard constraint(충돌 · 도로 경계) 구현과 합격/실격 판정 (TODO 8~10, 직접 해보기 ③)
6. Weight와 제약을 바꿔 가며 실패 case 복구 → 종합 미션 4/4 달성 (TODO 11~17)
7. 충돌을 제약 대신 벌금(cost)으로 막으면 어떻게 되는지 직접 확인 (TODO 18·19, 심화)
8. 도전 시나리오 3가지(S자 · 슬라럼 · 차선 봉쇄)에 같은 방법 적용 (TODO 20~22)

### 진행 방식

> **비용 함수의 weight를 바꾸거나 제약을 켤 때마다, 화면에 [트랙 그림 + 결과(PASS/FAIL)]가 같이 나옵니다.**
> 빨간 ✗ 가 초록 ✓ 로 바뀌도록 숫자와 제약 조건을 고쳐가며, optimization trajectory planning을 몸으로 익힙니다.

결과 4가지는 다음과 같습니다.

| 항목 | 의미 |
|---|---|
| **Tracking** | 초록 reference path를 잘 따라가는가 |
| **On-road** | 도로 밖으로 나가지 않는가 |
| **No-collision** | 장애물과 부딪히지 않는가 |
| **Comfort** | 급조향(조향 변화율 δ̇) 없이 부드러운가 |

**용어** (이 노트북은 한국어와 영어를 섞어 씁니다. 아래 짝은 같은 뜻입니다)

| 영어 (코드) | 한국어 | 뜻 |
|---|---|---|
| cost | 비용 | 궤적의 나쁜 정도 점수 (낮을수록 좋음) |
| weight | 가중치 | 비용 항목별 중요도 |
| constraint | 제약 | 반드시 지킬 조건 (어기면 실격) |
| trajectory | 궤적 | 경로 + 시간에 따른 움직임 (Step 02 정의) |
| soft / hard | 양보 가능 / 양보 불가 | cost 는 soft, constraint 는 hard |

---
# 0. Setup & 시나리오

- **Setup** : 라이브러리와 utils 모듈(상수 · 시나리오 생성 · solver 실행부 · 시각화)을 불러옵니다.
- **시나리오** : 이번 실습에서 사용할 reference path + 장애물 구성을 미리 확인합니다.


In [ ]:
%matplotlib inline
import sys, os
sys.path.insert(0, os.path.abspath('../utils/02_trajectory_planning'))

import numpy as np
import matplotlib.pyplot as plt

# 개념 학습에 직접 필요하지 않은 코드(상수, 시나리오 생성, solver 실행부, 그리기·채점판·시각화)는
# utils/02_trajectory_planning/step04_optimization_trajectory_planner.py 에 정의되어 있습니다.
from step04_optimization_trajectory_planner import (
    # 공통 상수
    LANE_W, NUM_LANES, ROAD_LEN, ROAD_Y0, ROAD_Y1, LANE,
    dt, N, L_WB, V_REF,
    R_EGO, R_OBJ, D_SAFE, ROAD_MARGIN, EGO_OFFSETS,
    MAX_STEER, MAX_DSTEER, STEER_WEIGHT, COMFORT_DSTEER,
    # 시나리오 (reference path + 장애물)
    MakeScenario, Thetaref, SCENARIO_DESC, EGO_START,
    # 기하 보조 (충돌 제약용 원 중심들)
    ObstacleCircleCenters, EgoCircleCenters,
    # solver (여러분이 완성한 함수들을 RegisterEngine 으로 연결해서 사용)
    RegisterEngine, Optimize, WarmStart,
    # 시각화 (그리기 · 채점판 · 프로파일 · 애니메이션)
    SetKoreanFont, PlotReferencePathPreview, PlotPlans,
    Evaluate, VisualizePlan, Run, ShowProfiles, Animate, ShowWeightSweep,
)

plt.rcParams['figure.dpi'] = 110
np.set_printoptions(precision=3, suppress=True)
SetKoreanFont()
print('준비 완료')


## 시나리오 — Reference Path와 장애물

- 이번 실습은 reference path를 가지고 **기본 시나리오 4가지 + 도전 시나리오 3가지** 를 연습합니다.

### 기본 시나리오 (이론 · 실습에서 사용)

| 시나리오 | reference path | 장애물 | 목적 |
|---|---|---|---|
| `curve` | L1→L3 부드러운 차선변경 | 없음 | reference를 잘 따라가는가 (추종) |
| `avoid` | L1 직진 유지 | 같은 차선 앞 정지차 | 직진 reference + 장애물 회피 |
| `badref` | 도로 밖으로 나가는 *나쁜* reference | 없음 | 나쁜 목표를 hard constraint가 막아주는가 |
| `mission` | L1→L3 차선변경 | L3 앞 정지차 | 추종 + 회피 + 안전을 한꺼번에 (종합) |

### 도전 시나리오 (노트북 마지막 '추가 학습'에서 사용)

| 시나리오 | reference path | 장애물 | 목적 |
|---|---|---|---|
| `scurve` | L1→L3→L1 S자 이중 차선변경 | 없음 | 어려운 reference 추종 ↔ comfort 트레이드오프 |
| `slalom` | L1 직진 유지 | L1·L2 엇갈린 정지차 2대 | 연속 회피 (슬라럼) 후 reference 복귀 |
| `wall` | L1 직진 유지 | L1·L2 나란한 정지차 2대 | 두 차선 동시 봉쇄 → reference를 잠시 포기하고 크게 우회 |

### Reference path 바로 확인하기

아래 그림은 **기본 시나리오 4가지**의 reference path를 보여줍니다 (도전 시나리오는 마지막에 다시 그려봅니다).

- **초록 점선** : reference path $y_{ref}(x)$ — 차가 따라가고 싶은 목표 선
- **빨간 표식** : 장애물 위치
- **파란 표식** : ego 차량의 시작 위치

> planner가 *무엇을 따라가려는지*, 그리고 *언제 reference와 안전 제약이 충돌하는지* 미리 감을 잡는 그림입니다.


In [ ]:
# [Setup] 시나리오 미리 보기 --------------------------------------------------
# 시나리오 생성 코드(LaneChangeReference, StraightReference, MakeScenario 등)는 utils 모듈에 있습니다.
# MakeScenario(name) 은 (Yref, Dyref, obstacles, x_view) 를 돌려줍니다.
print('사용 가능한 시나리오:')
for _name, _desc in SCENARIO_DESC.items():
    print(f'  {_name:8s} — {_desc}')

# 기본 시나리오 4가지의 reference path 미리 보기
PlotReferencePathPreview()


---
# Optimization Trajectory Planning 이론 흐름

**Trajectory planning이란?**
- 차가 앞으로 몇 초 동안 **어떤 궤적으로 움직일지**를 정하는 것
- 지도 위의 선 하나가 아니라, 시간에 따른 **위치·방향·속도** + 그것을 만드는 **가속도·조향**을 함께 결정

**최적화 문제로 보면:**

$$
\begin{aligned}
\min_{\mathbf{u}_{0:N-1}} \quad & J(\mathbf{x}_{0:N}, \mathbf{u}_{0:N-1}) \\
\text{s.t.} \quad
& \mathbf{x}_{k+1}=f(\mathbf{x}_k, \mathbf{u}_k) \\
& \mathbf{u}_{min} \le \mathbf{u}_k \le \mathbf{u}_{max}
\end{aligned}
$$

- **고른다** → control sequence $u_{0:N-1}$ (각 $u_k=\dot\delta_k$, 조향 변화율 / 속도는 고정 = 횡방향만 최적화)
- **만든다** → bicycle model이 그 control로 미래 trajectory 생성
- **점수 매긴다** → cost function $J$ 로 trajectory가 얼마나 좋은지 평가
- **찾는다** → solver가 *cost는 최소, constraint는 만족* 하는 해를 탐색

> **constraint(제약, s.t. = "단, 아래를 지키며")** =
> ① 차는 **bicycle model 대로만** 움직임 ($\mathbf{x}_{k+1}=f(\mathbf{x}_k,\mathbf{u}_k)$)  
> ② 조향각·조향 변화율이 **물리적 한계 안** ($\mathbf{u}_{min}\le\mathbf{u}_k\le\mathbf{u}_{max}$)  
> ③ **충돌·도로 이탈 금지**(안전). (자세한 건 4절)

**planning에 필요한 4가지 재료**

| 재료 | 역할 | 한 줄 요약 |
|---|---|---|
| **① Reference path** | 가고 싶은 목표선 | "이쪽으로 가고 싶다" (절대 명령 ✗) |
| **② Vehicle model** | control → trajectory 예측 | "이 조작이면 차가 여기로 간다" |
| **③ Cost function** | 좋은 궤적의 기준 (soft) | "경로 추종(y,θ) + 조향 부드럽게" |
| **④ Constraint** | 반드시 지킬 선 (hard) | "충돌·도로 이탈·제어 한계 초과 금지" |

cost function은 뒤에서 이렇게 생겼습니다 (자세한 건 3절):

$$
J = \underbrace{w_{track}\overline{(y-y_{ref})^2}+w_{heading}\overline{(\theta-\theta_{ref})^2}}_{\text{추종}}
  + \underbrace{w_{steer}\,\overline{\delta^2}}_{\text{조향각}}
  + \underbrace{w_{comfort}\,\overline{\dot\delta^2}}_{\text{control input = 조향 변화율}}
$$

**이 실습의 진행 순서**
1. reference path를 생성한다
2. bicycle model로 trajectory가 생성되는 원리를 본다
3. cost + constraint로 solver가 궤적을 만드는 과정을 본다
4. *일부러 실패하는 case* 로 weight와 constraint의 역할을 분리해 이해한다
5. 직접 실패 case를 고치고, 비용 항을 구현한다

---
# *실제 planning 코드*

지금까지 설명한 이론을 **그 순서 그대로 코드로** 옮긴 부분입니다.

**어떻게 보면 되나**
- 각 코드 셀은 앞 이론의 **한 단계와 1:1로 대응** 합니다.
- Reference path 와 시나리오는 맨 위 **0. Setup & 시나리오** 에서 이미 준비했습니다.
- "이론에서 본 개념이 코드로는 이렇게 생겼구나" 를 확인하는 게 목표 — 지금 코드를 외울 필요는 없습니다.

| 순서 | 코드 셀 | 이론에서 본 것 |
|---|---|---|
| 2 | Bicycle model | control → 미래 trajectory |
| 3 | Cost function | 좋은 궤적의 점수 (soft) |
| 4 | Hard constraint | 절대 지킬 선 (충돌·도로·제어한계) |
| 5 | Solver 실행 | 위를 연결해 최적 control 탐색 (코드는 utils — 역할만 이해하면 됩니다) |

> 단계 2·3·4 뒤에는 각각 **직접 해보기 ①~③** 이 이어집니다 — 조향 계획을 손으로 만들고(①), 채점하고(②),
> 실격을 판정합니다(③). 이 세 개가 "최적화가 무엇인지"의 핵심입니다.
>
> 차·도로를 **그리는 코드**, 그리고 **채점(scorecard) 코드** 는 개념 학습에 필요하지 않으므로
> `utils/02_trajectory_planning/step04_optimization_trajectory_planner.py` 모듈에 모아 두었고, 노트북에서는 import 해서 사용만 합니다.
> 지금은 **계획이 어떻게 계산되는지(아래 섹션 2~5)** 에만 집중하세요. 그림 코드는 결과를 눈으로 보기 위한 도구일 뿐입니다.

## 2. Kinematic Bicycle Model (횡방향 / lateral)

### 왜 이렇게 두는가?
- 이번 실습은 **속도는 고정($V_{REF}$)** 하고 **횡방향(조향)만** 최적화합니다.
- 그래서 **control input = 조향 변화율 $\dot\delta$** (핸들을 얼마나 빨리 돌릴지), **조향각 $\delta$ 는 state** 로 둡니다.
- (참고) 실제 차량의 LMPC(선형 모델 예측 제어)도 같은 구조입니다 (조향을 부드럽게 다루려고 $\delta$ 를 state, $\dot\delta$ 를 input 으로).

### State / Control
$$ \text{State}\quad \mathbf{x}=[\,x,\ y,\ \theta,\ \delta\,] \qquad\qquad \text{Control}\quad u=\dot\delta\ (\text{조향 변화율}) $$
( 속도 $v=V_{REF}$ 로 고정 )

### Model
$$
\begin{aligned}
\dot{x} &= v\cos\theta \\
\dot{y} &= v\sin\theta \\
\dot{\theta} &= \frac{v\tan\delta}{L_{WB}} \\
\dot{\delta} &= u
\end{aligned}
$$

### Optimization 과의 연결

1. Solver가 control input $\dot\delta$ 를 선택한다

2. Bicycle model이 미래 trajectory(조향각 $\delta$ 포함)를 예측한다

3. Cost function이 reference path 와 비교한다

4. 가장 cost가 작은 $\dot\delta$ 시퀀스가 선택된다

> **실습**: 아래 코드 셀의 `TODO 1` 블록을 채워 이 model 식을 완성하세요. 맞으면 **정답!** 이 뜹니다. 틀려도 아래 셀은 실행되지만 결과가 틀리므로, **정답!** 확인 후 진행.

In [ ]:
# [Theory 2] kinematic bicycle model (횡방향 / lateral) -------------------
# state   x = [x, y, θ, δ]   (위치, heading, 조향각)   * 속도 v 는 V_REF 로 고정
# control u = δ̇  (조향 변화율, steering rate)  ← solver 가 고르는 값(control input)
def KinematicBicycleStep(state, dsteer):
    x     = state[0]
    y     = state[1]
    theta = state[2]
    delta = state[3]                          # 조향각 (state)

    ##### TODO 1 #####
    # 'Δt(dt) 동안 한 스텝 전진' 완성하기: 다음값 = 현재값 + (변화율) × dt
    #   - next_x     : ẋ = v·cosθ
    #   - next_y     : ẏ = v·sinθ
    #   - next_theta : θ̇ = v·tanδ / L_WB
    #   - next_delta : δ̇ = dsteer (조향 변화율 입력)
    #   (속도 v = V_REF, 축간거리(휠베이스) L_WB = 2.7 m)
    # 사용함수: np.cos(rad) / np.sin(rad) / np.tan(rad) / 기능: 라디안 각도의 삼각비를 반환
    #################

    next_x     = ...
    next_y     = ...
    next_theta = ...
    next_delta = ...

    #################

    return np.array([next_x, next_y, next_theta, next_delta])


def Simulate(dsteer_sequence, initial_state):
    controls = np.asarray(dsteer_sequence).reshape(-1)   # δ̇ 시퀀스 (길이 N)
    states = np.zeros((len(controls) + 1, 4))
    states[0] = initial_state
    for step in range(len(controls)):
        states[step + 1] = KinematicBicycleStep(states[step], controls[step])
    return states


# TODO 를 채우고 이 셀을 실행하면 바로 결과가 표시됩니다.
try:
    _s0 = np.array([1.0, 2.0, 0.3, 0.10])                 # x=1, y=2, θ=0.3, δ=0.10
    _got = KinematicBicycleStep(_s0, dsteer=0.05)
    _ans = np.array([3.2928075739, 2.7092484960, 0.3891863752, 0.11])   # 기대값 (V_REF=12, dt=0.2, L_WB=2.7)
    names = ['next_x', 'next_y', 'next_theta', 'next_delta']
    if np.allclose(_got, _ans):
        print('정답! bicycle model 완성 -> 아래 셀 진행')
    else:
        print(' 아직 안 맞음 (❌ 항목만 수정):')
        for n, g, a in zip(names, _got, _ans):
            print(('   ✅ ' if np.isclose(g, a) else '   ❌ ') + f'{n:<10s} 내 결과={g:.6f} / 정답={a:.6f}')
except Exception as e:
    print('!!! 아직 TODO 1 블록이 완성되지 않았어요 →', e)

## 직접 해보기 ① — 결정변수 $z$(조향 계획)를 내 손으로

방금 완성한 bicycle model 덕분에, **조향 계획 $z$ 하나만 정하면 미래 trajectory 가 통째로 결정**됩니다.
연습 무대는 `avoid` 시나리오(L1 직진 reference + 앞에 정지차)이고, 계획 두 개를 만듭니다.

- 계획 1 `z_straight` : 아무것도 안 하는 직진 ($\dot\delta = 0$)
- 계획 2 `z_mine` : 손으로 만든 왼쪽 차선변경 — 핸들 세기 `S` 를 **여러분이** 정합니다

> 여기서 만든 두 계획은 이어지는 **직접 해보기 ②(채점) → ③(실격 판정)** 에서 계속 사용됩니다.


In [ ]:
# [직접 해보기 ①] 결정변수 z 만들기 ---------------------------------------
# z = 매 스텝(dt=0.2초)의 조향 변화율 δ̇ 을 담은 길이 N(=50)의 배열.

# 연습 무대: avoid 시나리오
Yref_avoid, Dyref_avoid, obstacles_avoid, xview_avoid = MakeScenario('avoid')
x0_avoid = np.array([12.0, LANE[1], 0.0, 0.0])          # 시작 상태 (L1 에서 직진 중)

# 계획 1: 아무것도 안 함 → 그냥 직진
z_straight = np.zeros(N)

# 계획 2: 손으로 만든 왼쪽 차선변경 (감기 → 반대로 풀기 → 직진 정렬)

##### TODO 2 #####
# 핸들 세기 S 정하기
#   - 0.01 ~ 0.15 사이에서 바꿔 가며 실행해 보세요.
#   - 너무 작으면 정지차를 못 피하고, 너무 크면 도로 밖으로 나갑니다. (0.03 ~ 0.10 사이면 성공)
#################

S = ...

#################

z_mine = np.zeros(N)
z_mine[0:5]   = +S            # 1초 동안 왼쪽으로 감고
z_mine[5:15]  = -S            # 2초 동안 반대로 풀고
z_mine[15:20] = +S            # 1초 동안 다시 감아 직진으로 정렬

# 두 계획을 trajectory 로 펼쳐 그려 보기  (z --Simulate--> 미래 궤적, 그리기는 utils 의 PlotPlans)
_s1 = Simulate(z_straight, x0_avoid)
_s2 = Simulate(z_mine, x0_avoid)
PlotPlans([(_s1, '계획 1: 직진 (z_straight)', '#8a8f96'),
           (_s2, '계획 2: 내 차선변경 (z_mine)', '#1f77b4')],
          scenario='avoid', title='결정변수 z 두 개 → 두 개의 미래 trajectory')


## 3. Cost Function — 무엇을 좋은 trajectory라고 볼 것인가 (횡방향)

### control input
- **control input** = $\dot\delta$ (조향 변화율) — solver 가 매 시점 고르는 값
- **조향각 $\delta$ 는 state** (현재 핸들 각도)

### 비용 항 (cost terms)

| 코드 weight | 코드 항 | 의미 |
|---|---|---|
| `track` | $\overline{(y-y_{ref})^2}$ | reference 에서 옆으로(y) 벗어남 |
| `heading` | $\overline{(\theta-\theta_{ref})^2}$ | 경로 방향과 어긋남 |
| `STEER_WEIGHT` (고정) | $\overline{\delta^2}$ | 조향각(state)을 불필요하게 크게 유지하지 않도록 |
| `comfort` | $\overline{\dot\delta^2}$ | **control input(조향 변화율)** 을 작게 = 부드러운 조향 |

전체 비용:
$$ J = w_{track}\overline{(y-y_{ref})^2} + w_{heading}\overline{(\theta-\theta_{ref})^2} + w_{steer}\overline{\delta^2} + w_{comfort}\overline{\dot\delta^2} $$

> 🔗 (참고, MPC를 아는 분) 실제 LMPC 대응: $w_{track},w_{heading},w_{steer}$ = **Q**(상태 추종/정규화), $w_{comfort}$ = **R**(입력 $\dot\delta$ 벌점).

> **cost는 soft(선호)** : 장애물이 reference 위에 있으면 추종을 조금 손해 보더라도 **충돌 constraint를 우선**한다.
> 튜닝 초점 = **track · heading · comfort** ( `STEER_WEIGHT` 는 고정 ).

> **실습**: 아래 코드 셀의 `TODO 3~6` 블록(각 비용 항)을 채우세요. 맞으면 **정답!** 이 뜹니다.

In [ ]:
# [Theory 3] cost function (횡방향) --------------------------------------
# 실제 LMPC 와 같은 구조:
#   tracking : (y - y_ref)²,  (θ - θ_ref)²          ← reference path 추종
#   조향각   : δ²   (state 정규화 — 핸들을 불필요하게 크게 유지하지 않도록)
#   조향변화 : δ̇²  (control input = 부드러운 조향 = comfort)

def TrackingCost(states, Yref):
    y_error = states[:, 1] - Yref(states[:, 0])   # 횡방향(y) 오차

    ##### TODO 3 #####
    # tracking 비용 = y_error 의 '평균 제곱(mean square)'
    # 사용함수: np.mean(x) / 기능: 배열 원소들의 산술 평균  (예: np.mean(error ** 2))
    #################

    cost = ...

    #################
    return cost

def HeadingCost(states, Dyref):
    theta_error = states[:, 2] - Thetaref(Dyref, states[:, 0])   # heading 오차

    ##### TODO 4 #####
    # heading 비용 = theta_error 의 평균 제곱
    #################

    cost = ...

    #################
    return cost

def SteerCost(states):
    ##### TODO 5 #####
    # 조향각 비용 = 조향각 δ(=states[:, 3]) 의 평균 제곱
    #################

    cost = ...

    #################
    return cost

def ComfortCost(controls):
    controls = np.asarray(controls)               # control input δ̇

    ##### TODO 6 #####
    # comfort 비용 = δ̇(=controls) 의 평균 제곱
    #################

    cost = ...

    #################
    return cost

def CostTermsFromControls(z, Yref, Dyref, initial_state):
    controls = np.asarray(z).reshape(-1)                 # δ̇ 시퀀스
    states = Simulate(controls, initial_state)
    terms = {
        'track':   TrackingCost(states, Yref),          # (y - y_ref)²
        'heading': HeadingCost(states, Dyref),          # (θ - θ_ref)²
        'steer':   SteerCost(states),                   # δ²  (조향각 크기)
        'comfort': ComfortCost(controls),               # δ̇² (조향 변화율 = control input)
    }
    return terms, states, controls

def CostFn(z, weights, Yref, Dyref, initial_state):
    terms, _, _ = CostTermsFromControls(z, Yref, Dyref, initial_state)
    return (
        weights['track']   * terms['track']
        + weights['heading'] * terms['heading']
        + STEER_WEIGHT       * terms['steer']            # 고정 weight (δ²)
        + weights['comfort'] * terms['comfort']          # δ̇²
    )

# TODO 를 채우고 이 셀을 실행하면 바로 결과가 표시됩니다.
try:
    _s = np.array([[0., 0.0, 0.00, 0.10],
                   [1., 0.5, 0.05, 0.20],
                   [2., 1.0, 0.10, 0.15]])               # 예시 states [x, y, θ, δ]
    _z = np.array([0.5, -0.25])                          # 예시 control δ̇
    _yr  = lambda x: 0.5 * np.ones_like(x)
    _dyr = lambda x: np.zeros_like(x)
    _ans = {'track': 0.1666666667, 'heading': 0.0041666667,         # 기대값
            'steer': 0.0241666667, 'comfort': 0.15625}
    _mine = {'track': TrackingCost(_s, _yr), 'heading': HeadingCost(_s, _dyr),
             'steer': SteerCost(_s), 'comfort': ComfortCost(_z)}
    if all(np.isclose(_mine[k], _ans[k]) for k in _ans):
        print('정답! cost function 완성 -> 아래 셀 진행')
    else:
        print(' 아직 안 맞아요:')
        for k in _ans:
            print('   [%s] %-8s 내 결과=%.5f / 정답=%.5f'
                  % ('OK' if np.isclose(_mine[k], _ans[k]) else 'X', k, _mine[k], _ans[k]))
except Exception as e:
    print('!!! 아직 TODO 3~6 블록이 완성되지 않았어요 →', e)

## 직접 해보기 ② — 방금 완성한 cost 로 내 계획 채점하기

cost function 은 **"계획 하나 → 점수 하나($J$, 낮을수록 좋음)"** 를 만들어 주는 채점기입니다.
서로 다른 계획도 점수로 바꾸면 **비교**할 수 있게 되고, 비교할 수 있어야 "가장 좋은 것 찾기(최적화)"가 가능해집니다.

①에서 만든 두 계획을 채점해 봅시다.


In [ ]:
# [직접 해보기 ②] 내 계획 채점하기 ----------------------------------------
# CostFn(z, weights, Yref, Dyref, x0) 은 계획 z 의 총점 J 를 돌려줍니다.
W_hand = {'track': 8, 'heading': 12, 'comfort': 8}      # 채점 기준 (weight)

##### TODO 7 #####
# 두 계획을 각각 채점하기: CostFn(계획 z, W_hand, Yref_avoid, Dyref_avoid, x0_avoid)
#   - J_straight : 계획 1 (z_straight) 의 점수
#   - J_mine     : 계획 2 (z_mine) 의 점수
#################

J_straight = ...
J_mine     = ...

#################

print(f'계획 1 (직진)        J = {J_straight:8.3f}')
print(f'계획 2 (내 차선변경)  J = {J_mine:8.3f}')

# 직진이 J=0.000, 만점입니다! reference(L1 직진)를 완벽히 따라가고 조향도 전혀 안 하니까요.
# ...그런데 ① 의 그림을 다시 보세요. 직진 계획은 정지차를 그대로 뚫고 지나갑니다.
# → cost 는 충돌을 전혀 모릅니다. 이것이 바로 다음 단계 hard constraint 가 필요한 이유입니다.


## 4. Hard Constraints — 절대 어기면 안 되는 조건

- **왜 cost로는 부족한가?**
  - 안전을 전부 cost(벌점)에 맡기면, solver가 *"조금 부딪혀도 다른 cost가 낮으니 OK"* 라는 해를 고를 수 있음 (→ 노트북 끝 **(심화)** 에서 직접 확인)
  - 그래서 안전은 **hard constraint로 분리**

**① 충돌 회피**
- ego·장애물을 **여러 개의 원**으로 근사 → 모든 시간 step에서 원 사이 거리 ≥ 안전거리
$$ d_{collision}=d-(r_{ego}+r_{obj})-d_{safe}\ge 0 $$
- $d$ = ego 원과 장애물 원 중심 거리 / 0보다 작아지면 **너무 가까움 → 금지**

**② 도로 경계**
- 차량 대표점이 도로 아래·위 경계 안에 있어야 함
$$ y_{ego}-y_{road,min}\ge0,\qquad y_{road,max}-y_{ego}\ge0 $$

**③ Control bound**
- 조향각 $|\delta| \le$ `MAX_STEER`, 조향 변화율 $|\dot\delta| \le$ `MAX_DSTEER` (이번 실습은 속도 고정이라 가속도 입력은 없음)

> **핵심**
> - **Soft (양보 가능)** : reference 추종, comfort
> - **Hard (양보 불가)** : 충돌, 도로 이탈, 제어 한계
>
> 이 구분이 optimization trajectory planning의 핵심입니다.

> **실습**: 아래 코드 셀의 `TODO 8·9` 블록(충돌 회피·도로 경계)을 채우세요. 맞으면 **정답!** 이 뜹니다.

In [ ]:
# [Theory 4] hard constraints --------------------------------------------
# 충돌 회피 / 도로 경계는 'soft cost' 가 아니라 '반드시 지켜야 하는 hard 제약'.
# SciPy SLSQP 의 ineq 제약 규칙:  fun(z) >= 0  이면 만족.
#   → 모든 제약 값을 '여유(margin) >= 0' 형태로 만든다.
# 차량을 근사하는 원들의 중심 계산(ObstacleCircleCenters / EgoCircleCenters)은 utils 에서 import 했습니다.

# (제약 1) 충돌 회피:모든 (ego원 × 장애물원 × 시점) 에서  '중심거리 - 최소허용거리 >= 0'
def CollisionConstraintValues(states, obstacles):
    if not obstacles:
        return np.array([1.0])
    constraint_values = []
    minimum_allowed_distance = R_EGO + R_OBJ + D_SAFE       # ego반지름 + 장애물반지름 + 안전여유
    for ego_centers in EgoCircleCenters(states):
        for obstacle_x, obstacle_y in ObstacleCircleCenters(obstacles):
            dx = ego_centers[:, 0] - obstacle_x          # 두 원 중심의 x 방향 거리차 (delta x)
            dy = ego_centers[:, 1] - obstacle_y          # 두 원 중심의 y 방향 거리차 (delta y)

            ##### TODO 8 #####
            # 충돌 회피 여유 계산하기
            #   - distance : 두 원 중심거리 = √(dx²+dy²)
            #   - 여유     : 중심거리 − 최소허용거리(minimum_allowed_distance)
            # 사용함수: np.sqrt(x) / 기능: 제곱근
            #################

            distance = ...
            constraint_values.append(...)

            #################
    return np.concatenate(constraint_values)

# (제약 2) 도로 경계: ego 대표점의 y 가 도로 안에  ->  '아래 여유 >= 0' 그리고 '위 여유 >= 0'
def RoadBoundaryConstraintValues(states):
    constraint_values = []
    road_lower_limit = ROAD_Y0 + ROAD_MARGIN
    road_upper_limit = ROAD_Y1 - ROAD_MARGIN
    for ego_centers in EgoCircleCenters(states):
        ego_y = ego_centers[:, 1]

        ##### TODO 9 #####
        # 도로 경계 여유 2개를 constraint_values 에 추가하기
        #   - 아래 경계 여유 = ego_y − 아래한계(road_lower_limit)
        #   - 위   경계 여유 = 위한계(road_upper_limit) − ego_y
        #################

        constraint_values.append(...)
        constraint_values.append(...)

        #################
    return np.concatenate(constraint_values)

# (제약 3) 조향각 한계: |δ| <= MAX_STEER   (state bound, 항상 적용) — 이미 완성됨
def SteerLimitConstraintValues(states):
    delta = states[:, 3]
    return np.concatenate([MAX_STEER - delta, MAX_STEER + delta])

def ConsFn(z, obstacles, initial_state, use_collision, use_road):
    states = Simulate(z, initial_state)                  # control(δ̇) -> trajectory
    constraints = [SteerLimitConstraintValues(states)]   # 조향각 한계(항상)
    if use_collision:
        constraints.append(CollisionConstraintValues(states, obstacles))
    if use_road:
        constraints.append(RoadBoundaryConstraintValues(states))
    return np.concatenate(constraints)

# TODO 를 채우고 이 셀을 실행하면 바로 결과가 표시됩니다.
try:
    _st  = np.array([[20., 4.0, 0.0, 0.0],
                     [21., 4.2, 0.0, 0.0]])               # 예시 trajectory [x, y, θ, δ]
    _obs = [(25., 6.0)]
    # 기대값: 충돌 여유 12개 (ego 원 2 × 장애물 원 3, 시점 2개) -> 최소 -1.203, 합 3.999
    #         도로 여유  8개 (ego 원 2 × 경계 2,   시점 2개) -> 최소 3.000, 합 62.000
    _col  = CollisionConstraintValues(_st, _obs)
    _road = RoadBoundaryConstraintValues(_st)
    _ok_c = _col.shape == (12,) and np.isclose(_col.min(), -1.2026334) and np.isclose(_col.sum(), 3.9988618)
    _ok_r = _road.shape == (8,) and np.isclose(_road.min(), 3.0) and np.isclose(_road.sum(), 62.0)
    if _ok_c and _ok_r:
        print('정답! 제약(충돌·도로) 완성 -> 아래 셀 진행')
    else:
        print(' 아직 안 맞아요:')
        print('   [%s] 충돌 회피  내 결과: %d개, 최소 %.3f, 합 %.3f / 기대: 12개, 최소 -1.203, 합 3.999'
              % ('OK' if _ok_c else 'X', _col.size, _col.min(), _col.sum()))
        print('   [%s] 도로 경계  내 결과: %d개, 최소 %.3f, 합 %.3f / 기대: 8개, 최소 3.000, 합 62.000'
              % ('OK' if _ok_r else 'X', _road.size, _road.min(), _road.sum()))
except Exception as e:
    print('!!! 아직 TODO 8·9 블록이 완성되지 않았어요 →', e)

## 직접 해보기 ③ — 실격 판정: 점수가 좋아도 안 되는 계획

방금 완성한 제약 함수들은 계획의 모든 시점에 대해 **여유(margin)** 를 계산합니다.
여유가 **하나라도 음수면 그 계획은 실격**입니다. ②에서 만점을 받았던 직진 계획을 판정해 봅시다.


In [ ]:
# [직접 해보기 ③] 실격 판정 ------------------------------------------------
# ConsFn 은 모든 제약(조향 한계 + 충돌 + 도로)의 여유(margin) 목록을 돌려줍니다.
margins_straight = ConsFn(z_straight, obstacles_avoid, x0_avoid, use_collision=True, use_road=True)
margins_mine     = ConsFn(z_mine,     obstacles_avoid, x0_avoid, use_collision=True, use_road=True)

##### TODO 10 #####
# 합격 판정 완성하기: 최악의 여유(margin 의 최솟값)가 0 이상이면 합격
#################

ok_straight = ...
ok_mine     = ...

#################

print(f'계획 1 (직진)        최악 여유 = {margins_straight.min():+6.2f} m  →  {"합격" if ok_straight else "실격 (충돌!)"}')
print(f'계획 2 (내 차선변경)  최악 여유 = {margins_mine.min():+6.2f} m  →  {"합격" if ok_mine else "실격"}')

# 정리: J 만점(0점)인 직진은 '실격', J 가 나빠도 내 차선변경은 '합격'.


## 5. Solver — 비용과 제약을 풀어 최적 control 찾기

- **여러 control input(조향 변화율) 조합을 자동으로 바꿔 가며, cost 가 가장 작고 제약을 만족하는 조합을 찾아주는 도구**
- 우리가 weight 를 정하면, solver 는 그 weight 아래 *최적의 조향* 을 대신 찾아 준다.
- 이번 실습은 **SciPy 의 `SLSQP`** solver 를 사용한다.

solver 내부는 이 실습의 범위가 아닙니다 — 실행 코드는 utils 모듈에 있습니다.
아래 셀은 여러분이 위에서 완성한 **model · cost · constraint 를 solver 에 연결**하고, 한 번 실행해 볼 뿐입니다.


In [ ]:
# [Theory 5] solver 실행 — solver 코드는 utils 에 있습니다. 여기서는 '연결하고 실행'만 합니다.
# 여러분이 완성한 model / cost / constraint 를 solver 에 연결:
RegisterEngine(KinematicBicycleStep, Simulate, CostFn, ConsFn)

# mission 시나리오를 한 번 풀어 보기
_demo = Optimize(weights={'track': 10, 'heading': 10, 'comfort': 20}, scenario='mission')
PlotPlans([(_demo['s'], 'solver 결과 (최적 trajectory)', '#1f77b4')],
          scenario='mission', title='SLSQP 가 찾은 최적 trajectory (mission 시나리오)')


---
# 핵심 정리 (시연 전 복습)

전체 planning 흐름을 한 번에 다시 묶으면:

1. **Reference path를 받는다**
   - 목표이지만 절대 명령은 아님 → 안전하지 않으면 **잠시 포기** 할 수 있어야 함
2. **Vehicle model로 미래를 예측한다**
   - bicycle model 덕분에 solver의 trajectory가 최소한 *"차처럼 움직이는"* 형태가 됨
3. **Cost function으로 trajectory를 평가한다**
   - tracking(경로 이탈) + heading(방향 차이) + comfort(급조작) → weight로 우선순위 결정
4. **Hard constraint로 안전을 강제한다**
   - 충돌·도로 경계는 *"벌점 많은 항"* 이 아니라 *"절대 넘으면 안 되는 선"*

**강사 실습 시연 (실패 case)**

| 실패 case | 결과 |
|---|---|
| tracking weight 없음 | reference 무시하고 직진 |
| comfort weight 너무 작음 | 움직임이 거칠어짐 |
| collision constraint 끔 | 장애물을 뚫고 감 |
| road boundary constraint 끔 | 잘못된 reference 따라 도로 밖으로 |

> **수강생 실습의 목표**
> 숫자를 외우는 게 아니라 →
> **어떤 문제는 cost weight로 고치고, 어떤 문제는 hard constraint로 막아야 하는지** 를 구분하는 것

---
# 미션 살펴보기

`mission` 시나리오입니다. **초록 점선**(reference)은 L1→L3로 부드럽게 차선변경하는 경로이고,
그 끝 **L3에 빨간 정지차**가 있습니다. 좋은 planner라면: 초록선을 따라가다가, 빨간차 앞에서
살짝 비켜 안전거리를 확보하고, 부드럽게 — 즉 **결과 4개 모두 초록**이어야 합니다.

먼저 "잘 된 정답"이 어떤 모습인지 한 번 봅시다.

In [ ]:
# 잘 튜닝된 weight 예시 (정답 중 하나)
W_good = {'track':10, 'heading':10, 'comfort':100}
res = Run(W_good, scenario='mission', title='well-tuned example')

위처럼 파란 궤적이 초록선을 따라가다 빨간차를 살짝 돌아 피하면 성공입니다.
이게 어떻게 가능한지, 이제 **일부러 망가뜨려** 가며 **weight와 제약의 역할**을 봅니다.

---
# 실패 case (강사 시연)

각 셀은 **weight(①②)** 또는 **제약(③④)** 하나를 **잘못** 줍니다.
- 볼 것: weight로 고칠 문제인가, 제약으로 막을 문제인가

### 실패 ①  *"reference를 무시한다"* — track·heading weight ≈ 0
추종 벌점이 없으니 planner는 초록 곡선을 따라갈 이유가 없습니다. 그냥 직진해 버립니다.

In [ ]:
res = Run({'track':0, 'heading':0, 'comfort':2}, scenario='curve', title='no tracking weight')

**중요**: `track`/`heading` 은 *"초록선을 따라가라"* 는 지시. 0이면 경로를 완전히 무시 → **Tracking ✗**.

### 실패 ②  *"휙휙 꺾는다"* — comfort weight 너무 작음
부드러움 벌점이 거의 없으니 핸들을 급하게(조향 변화율 $\dot\delta$ 가 큼) 휙휙 꺾어 조향이 톱니처럼 됩니다.

In [ ]:
res = Run({'track':10, 'heading':8, 'comfort':0.0}, scenario='mission', title='comfort too low')
ShowProfiles(res)

**중요**: `comfort` 가 너무 작으면 승차감이 끔찍해집니다 → **Comfort ✗** (조향 변화율 $|\dot\delta|$ 이 큼).

### 실패 ③  *"장애물을 뚫고 간다"* — 충돌 제약을 꺼버림
충돌은 **hard 제약**입니다. 이걸 끄면(`use_collision=False`) planner는 초록선을 충실히 따라
빨간차를 그대로 관통합니다.

In [ ]:
res = Run({'track':8, 'heading':12, 'comfort':8}, scenario='avoid',
          use_collision=False, title='collision constraint OFF')

**중요**: 충돌 회피는 비용으로 부탁하는 게 아니라 **제약으로 금지**해야 안전합니다 → **No-collision ✗**.

### 실패 ④  *"나쁜 목표를 곧이곧대로 따라간다"* — reference가 도로 밖으로
`badref` 는 reference 자체가 도로 밖을 향합니다. 도로 경계 제약을 **끄면** 차는 초록선을 믿고
트랙 밖으로 나가버립니다.

In [ ]:
res = Run({'track':15, 'heading':8, 'comfort':8}, scenario='badref',
          use_road=False, title='bad reference, NO road boundary')

### 실패 ④ 의 해결 — 도로 경계를 **제약으로** 켜기
같은 나쁜 reference라도, 도로 경계 제약을 켜면(`use_road=True`) planner는
*"초록선을 따라가고 싶지만 도로는 못 벗어난다"* 며 트랙 안에 머뭅니다.

In [ ]:
res = Run({'track':15, 'heading':8, 'comfort':8}, scenario='badref',
          use_road=True, title='bad reference, road boundary ON (constraint protects)')

**중요**: **hard 제약은 나쁜 목표(soft)로부터 우리를 지켜줍니다.** 핵심은 On-road 가 ✗→✓ 로 바뀌는 부분입니다.
reference 자체가 도로 밖에 있으니 Tracking 은 ✗ 로 남습니다. 따라갈 수 없는 목표는 도로 경계에서 포기한 것 — planner가 soft 목표보다 hard safety를 우선한다는 점을 확인하세요.


---
# Weight를 연속적으로 바꾸면 — trajectory의 점진적 변화

위 실패 case 들은 weight 를 극단적으로(0 으로) 바꿨지만, 실제 튜닝에서는 weight 를 조금씩 바꿉니다.
weight 가 변하면 최적 trajectory 도 **연속적으로 조금씩** 변합니다.

아래 그림은 어려운 S자 차선변경(`scurve`)에서 weight 하나만 키워 가며 최적화를 반복한 결과입니다.
**색이 진할수록(보라) weight 가 작고, 밝을수록(노랑) 큰 결과**입니다.

- **위** : 도로 위 trajectory — 언뜻 보면 다 비슷해 보이지만,
- **아래 왼쪽** : reference 와의 횡방향 편차 $y - y_{ref}$ — weight 에 따라 **점진적으로** 벌어지고/붙는 것이 보입니다
- **아래 오른쪽** : 조향 변화율 $\dot\delta$ — 부드러움(comfort)이 어떻게 변하는지 보입니다

정리하면:

- `comfort` ↑ : 조향은 점점 부드러워지는 대신, reference 편차가 점점 커진다
- `track` ↑ : reference 에 점점 바짝 붙는 대신, 조향이 점점 급해진다

진행 방식:

1. 첫 번째 sweep(`comfort` ↑)은 **함께 결과를 봅니다.**
2. 두 번째 sweep(`track` ↑)은 **여러분이 결과를 먼저 예측**하고, `TODO 11` 블록을 채워 확인합니다.

> 튜닝은 "정답 하나 찾기"가 아니라 **트레이드오프의 균형점 찾기** 라는 것을 눈으로 확인해 봅시다.


In [ ]:
# ShowWeightSweep 은 utils 모듈에 정의되어 있습니다 — weight 하나만 바꿔 가며 최적화를 반복합니다.
ShowWeightSweep('comfort', [1, 20, 80, 300],
                base_weights={'track': 10, 'heading': 10, 'comfort': 20},
                scenario='scurve',
                title='comfort ↑ — 조향은 점점 부드럽게, reference 편차는 점점 크게 (scurve)')


In [ ]:
##### TODO 11 #####
# 위 comfort sweep 을 참고해, 이번에는 'track' 을 키워 가며 sweep 해 봅니다.
# 실행하기 전에 먼저 예측해 보세요: track 을 0.5 → 40 으로 키우면
#   (1) reference 편차는?  (2) 조향 변화율은?
#   (hint) comfort 와 반대 방향의 트레이드오프가 나타나야 합니다.
#################

ShowWeightSweep(...,                    # 키워 볼 weight 이름 (문자열)
                [...],                  # 바꿔 볼 값 4개 (작은 값 -> 큰 값)
                base_weights={'track': 10, 'heading': 0, 'comfort': 40},
                scenario='scurve',
                title='track ↑ (scurve) — 결과를 먼저 예측해 보세요')

#################

# 예측과 결과가 같았나요? 편차는 줄고(reference 에 붙고), 조향은 급해졌다면 정확히 이해한 것입니다.


---
# 수강생 실습 — 실패 Case를 직접 고쳐보기

이제 강사가 보여준 실패 상황을 수강생이 case별로 다시 고쳐 봅니다.
각 셀의 TODO 블록(TODO 12~16)을 숫자 또는 `True`/`False`로 채운 뒤 실행하세요.

진행 방식은 다음과 같습니다.

1. 실패 원인을 하나만 수정
2. `Run()`을 실행해 결과에서 어떤 항목이 바뀌는지 확인
3. `cost`로 고칠 문제인지, `constraint`로 막아야 할 문제인지 판단

추천값을 바로 맞히는 것보다, 값을 조금씩 바꾸며 빨간 항목이 왜 바뀌는지 보는 것이 더 중요합니다! 


## 실습 Case 1 — reference를 따라가도록 weight 수정하기

강사 시연의 실패 ①과 같은 상황입니다. `track`, `heading`이 0이면 초록 reference를 따라갈 이유가 없습니다.
`curve` 시나리오에서 Tracking이 초록이 되도록 두 weight를 채워 보세요.


In [ ]:
##### TODO 12 #####
# W_case1 채우기: reference 를 따라가게 만드는 두 weight (숫자, 예: 5, 10, 15 ...)
#   - track   : reference 와의 y 오차를 줄이는 힘
#   - heading : reference 방향과 정렬하는 힘
#   (comfort 는 0 으로 고정)
#################

W_case1 = {
    'track':    ...,
    'heading':  ...,
    'comfort':  0.0,
}

#################

res_case1 = Run(W_case1, scenario='curve', title='Student Case 1: fix tracking')


## 실습 Case 2 — 휙휙 꺾는 planner 부드럽게 만들기

강사 시연의 실패 ②와 같은 상황입니다. `comfort`가 너무 작으면 solver가 조향을 급하게(δ̇ 크게) 씁니다.
이번에는 `track`, `heading`은 고정하고 `comfort`만 조절해 Comfort가 초록이 되도록 해보세요.


In [ ]:
##### TODO 13 #####
# W_case2 의 comfort 정하기 (숫자)
#   (hint) 작으면 조향이 급격히 변하고(Comfort ✗), 키울수록 부드러워지는 대신 reference 편차(RMSE)가 커짐
#   확인: 채점판 Comfort 의 max|dsteer| < 1.20 rad/s 이면 초록 (comfort 0 -> 2.76, 1 -> 1.87 rad/s)
#################

W_case2 = {
    'track':    10.0,
    'heading':  8.0,
    'comfort':  ...,
}

#################

res_case2 = Run(W_case2, scenario='mission', title='Student Case 2: fix comfort')
ShowProfiles(res_case2)


## 실습 Case 3 — 장애물을 뚫고 가는 planner 막기

강사 시연의 실패 ③과 같은 상황입니다. 충돌 회피는 cost가 아니라 hard constraint로 막아야 합니다.
아래 빈칸을 채워 `No-collision`을 초록으로 바꾸세요.


In [ ]:
W_case3 = {'track': 8.0, 'heading': 12.0, 'comfort': 8}

##### TODO 14 #####
# use_collision 정하기 (True / False): 장애물을 뚫고 가지 않게 하려면?
#   - True : 충돌 제약 켬 -> 장애물 회피
#   - False: 충돌 제약 끔 -> reference 따라 장애물 관통
#################

use_collision = None

#################
assert use_collision in (True, False), 'use_collision 미입력: True 또는 False 로 채우기'

res_case3 = Run(W_case3, scenario='avoid', use_collision=use_collision,
                title='Student Case 3: collision constraint')


## 실습 Case 4 — 도로 밖 reference가 들어왔을 때 제약 없이 재현하기

강사 시연의 실패 ④를 먼저 직접 재현합니다. `badref`는 reference 자체가 도로 밖으로 향합니다.
도로 경계 constraint를 끄면 어떤 항목이 빨개지는지 확인하세요.


In [ ]:
W_case4 = {'track': 15.0, 'heading': 8.0, 'comfort': 8}

##### TODO 15 #####
# use_road 정하기 (True / False): 강사 실패 상황(도로 밖으로 나감)을 재현하려면?
#   - True : 도로 경계 제약 켬 -> 도로 안에 머묾
#   - False: 도로 경계 제약 끔 -> reference 따라 도로 밖으로
#################

use_road = None

#################
assert use_road in (True, False), 'use_road 미입력: True 또는 False 로 채우기'

res_case4 = Run(W_case4, scenario='badref', use_road=use_road,
                title='Student Case 4: bad reference without road constraint')


## 실습 Case 5 — 같은 bad reference를 hard constraint로 보호하기

이번에는 같은 `badref`에서 도로 경계 constraint를 켭니다.
목표 자체가 나쁘기 때문에 Tracking은 여전히 실패하지만, On-road가 어떻게 바뀌는지 확인하세요.
이 case는 **soft cost가 틀렸을 때 hard constraint가 안전을 지키는 역할**을 보여줍니다.


In [ ]:
W_case5 = {'track': 15.0, 'heading': 8.0, 'comfort': 8}

##### TODO 16 #####
# use_road 정하기 (True / False): 같은 bad reference 에서 도로 안에 머무르게 하려면?
#   - True : 도로 경계 제약 켬 -> 도로 안에 머묾
#   - False: 도로 경계 제약 끔 -> reference 따라 도로 밖으로
#################

use_road = None

#################
assert use_road in (True, False), 'use_road 미입력: True 또는 False 로 채우기'

res_case5 = Run(W_case5, scenario='badref', use_road=use_road,
                title='Student Case 5: road constraint protects')


---
# 종합 미션 — 비용과 제약을 함께 맞추기

마지막으로 `mission` 시나리오에서 4/4 초록을 만드세요.
이번 셀에서는 weight 3개뿐 아니라, 충돌 제약과 도로 제약이 켜져 있어야 하는지도 직접 채웁니다.


In [ ]:
##### TODO 17 #####
# 종합 미션: weight 3개와 제약 2개를 모두 채워 4/4 초록 만들기
#   - track / heading / comfort : 숫자
#   - use_collision / use_road  : True 또는 False
#################

W_final = {
    'track':    ...,
    'heading':  ...,
    'comfort':  ...,
}
use_collision = None
use_road = None

#################
assert use_collision in (True, False), 'use_collision 미입력: True 또는 False 로 채우기'
assert use_road in (True, False), 'use_road 미입력: True 또는 False 로 채우기'

res = Run(W_final, scenario='mission',
          use_collision=use_collision,
          use_road=use_road,
          title='Student Final Mission')


4/4 초록이 떴다면 아래 셀로 제어 프로파일도 확인해 보세요.
조향 변화율($\dot\delta$)과 조향각($\delta$) 그래프가 너무 날카로우면 `comfort`를 조금 키우고, reference를 못 따라가면 `track` 또는 `heading`을 조절해 보세요.


In [ ]:
ShowProfiles(res)
# 주행 애니메이션도 보고 싶다면 아래 주석을 푸세요:
# Animate(res)


---
# (심화) 충돌을 제약 대신 벌금(cost)으로 막으면?

섹션 4 에서 "안전을 전부 cost(벌점)에 맡기면, solver 가 *조금 부딪혀도 다른 cost 가 낮으니 OK* 라는 해를 고를 수 있다"고 했습니다.
이 말이 사실인지 **직접 확인**합니다. 충돌 제약을 끄고, 대신 장애물에 가까이 가면 내는 **벌금(벌점 항)** 을 비용에 더합니다.

- 제약(hard) = 장애물 주변 **출입 금지**
- 비용(soft) = 장애물 가까이 가면 **벌금**
- solver = 총비용이 가장 싼 계획을 고르는 계산기 → 벌금이 싸면?

**벌점**: 최소 허용거리 $d_{min}$ 보다 가까운 만큼(intrusion)의 제곱 평균 (멀면 0)

$$ \text{벌점} = \overline{\max(0,\ d_{min} - d)^2}, \qquad d_{min} = r_{ego} + r_{obj} + d_{safe} = 3.1\text{ m} \quad(\text{TODO 8 과 같은 값}) $$

| 원 중심거리 $d$ | $d_{min}$ 보다 가까운 만큼 | 벌점 |
|---|---|---|
| 4.0 m | 0 (충분히 멂) | 0 |
| 3.1 m | 0 (경계) | 0 |
| 2.6 m | 0.5 m | 0.25 |
| 1.1 m | 2.0 m | 4.0 |

**벌금 버전 총비용**: $J_{soft} = J + w_{obstacle}\cdot\text{벌점}$  ($J$ = 섹션 3 의 `CostFn`)

> **실습**: 아래 코드 셀의 `TODO 18·19` 블록을 채우세요. 맞으면 **정답!** 이 뜹니다. 그다음 실험 셀에서 벌금 weight 를 1 → 10 → 100 으로 바꿔 결과를 비교합니다.

In [ ]:
# [심화] 충돌을 벌금(soft cost)으로 --------------------------------------------
# 주의: mission 시나리오 전용 (CostFn 인자에 장애물이 없어서 mission 의 장애물 목록을 고정해 사용)
Yref_mission, Dyref_mission, obstacles_mission, _ = MakeScenario('mission')
x0_mission = np.array([12.0, LANE[1], 0.0, 0.0])          # 시작 상태 (다른 시나리오와 같음)

def SoftCollisionCost(states, obstacles):
    minimum_allowed_distance = R_EGO + R_OBJ + D_SAFE     # 3.1 m (TODO 8 과 같은 값)
    penalty = 0.0
    for ego_centers in EgoCircleCenters(states):
        for obstacle_x, obstacle_y in ObstacleCircleCenters(obstacles):
            distance = np.sqrt((ego_centers[:, 0] - obstacle_x)**2 + (ego_centers[:, 1] - obstacle_y)**2)

            ##### TODO 18 #####
            # 벌점 누적하기: 최소허용거리보다 가까운 만큼(intrusion)의 제곱 평균, 멀면 0
            #   - intrusion : minimum_allowed_distance − distance, 음수(충분히 멂)는 0 으로 자르기
            #   - penalty   : 지금까지의 penalty + intrusion² 의 평균
            # 사용함수: np.maximum(a, b) / 기능: 원소별 큰 값 (np.maximum(0.0, x) -> 음수를 0 으로)
            # 주의: 파이썬 내장 max(0, 배열) -> 에러 (배열은 np.maximum)
            #################

            intrusion = ...
            penalty = ...

            #################
    return penalty

def SoftCollisionCostFn(z, weights, Yref, Dyref, initial_state):
    states = Simulate(z, initial_state)

    ##### TODO 19 #####
    # 벌금 버전 총비용 만들기: 기존 비용 + 벌금 weight × 충돌 벌점
    #   - 기존 비용  : CostFn(z, weights, Yref, Dyref, initial_state)
    #   - 벌금 weight: weights['obstacle']
    #   - 충돌 벌점  : SoftCollisionCost(states, obstacles_mission)
    #################

    total = ...

    #################
    return total

# TODO 를 채우고 이 셀을 실행하면 바로 결과가 표시됩니다.
try:
    _st = np.array([[20.0, 4.0, 0.0, 0.0]])                    # ego 1 시점: x=20, y=4, 직진
    _far  = SoftCollisionCost(_st, [(60.0, 4.0)])              # 40 m 앞 장애물
    _near = SoftCollisionCost(_st, [(25.0, 4.0)])              # 5 m 앞 장애물 (원 일부가 겹침)
    _z = WarmStart(x0_mission, Yref_mission, Dyref_mission)    # reference 를 따라 정지차 쪽으로 가는 계획
    _total = SoftCollisionCostFn(_z, {'track': 8, 'heading': 12, 'comfort': 8, 'obstacle': 10},
                                 Yref_mission, Dyref_mission, x0_mission)
    _checks = [('먼 장애물 벌점 = 0',          np.isclose(_far, 0.0),     'TODO 18: 음수 intrusion 을 0 으로 자르기 (np.maximum)'),
               ('가까운 장애물 벌점 = 11.29',  np.isclose(_near, 11.29),  'TODO 18: intrusion 의 제곱 평균 누적 (제곱 누락 -> 4.70)'),
               ('벌금 버전 총비용 = 10.451',   np.isclose(_total, 10.451, atol=1e-3), "TODO 19: CostFn + weights['obstacle'] × 벌점")]
    if all(ok for _, ok, _ in _checks):
        print('정답! 벌금 비용 완성 -> 아래 실험 셀 진행')
    else:
        print(' 아직 안 맞아요:')
        for name, ok, hint in _checks:
            print(f"   [{'OK' if ok else 'X'}] {name}" + ('' if ok else f'  -> {hint}'))
except Exception as e:
    print('!!! 아직 TODO 18·19 블록이 완성되지 않았어요 →', e)

In [ ]:
# [심화 실험] mission, 충돌 제약 OFF, 벌금 weight 만 1 -> 10 -> 100 -------------------
W_soft = {'track': 8, 'heading': 12, 'comfort': 8}
assert np.isscalar(SoftCollisionCostFn(np.zeros(N), {**W_soft, 'obstacle': 1}, Yref_mission, Dyref_mission, x0_mission)), \
    'TODO 18·19 미완성: 위 셀에서 정답! 확인 후 실행'
RegisterEngine(KinematicBicycleStep, Simulate, SoftCollisionCostFn, ConsFn)   # 엔진의 비용을 '벌금 버전'으로 교체
try:
    res_soft = {w: Run({**W_soft, 'obstacle': w}, scenario='mission', use_collision=False,
                       title=f'충돌을 벌금으로: obstacle weight = {w}')
                for w in [1, 10, 100]}
finally:
    RegisterEngine(KinematicBicycleStep, Simulate, CostFn, ConsFn)            # 원래 비용으로 복구 (아래 도전 셀용)

# 비교 기준: 충돌 제약 ON = 부딪히지 않는 계획 중 가장 싼 것 (안전 여유를 지키므로 벌점 0)
res_safe = Optimize(weights=W_soft, scenario='mission', use_collision=True)
J_safe = CostFn(res_safe['u'], W_soft, Yref_mission, Dyref_mission, x0_mission)

print('같은 벌금 비용으로 매긴 총 J (낮을수록 solver 가 선호)')
print(f"{'계획':<16}{'clear':>9}{'기존 비용':>10}{'벌금':>8}{'총 J':>8}")
for w, r in res_soft.items():
    base = CostFn(r['u'], W_soft, Yref_mission, Dyref_mission, x0_mission)
    fine = w * SoftCollisionCost(r['s'], obstacles_mission)
    clear = Evaluate(r)['No-collision'][1].replace('clear=', '')
    note = '  <- 안전 계획보다 쌈' if base + fine < J_safe else ''
    print(f"{f'벌금 weight {w}':<16}{clear:>9}{base:10.2f}{fine:8.2f}{base + fine:8.2f}{note}")
clear_safe = Evaluate(res_safe)['No-collision'][1].replace('clear=', '')
print(f"{'충돌 제약 ON':<16}{clear_safe:>9}{J_safe:10.2f}{0.0:8.2f}{J_safe:8.2f}")

**결과 읽기** (clear = 두 원 사이의 실제 틈, 음수면 겹침 = 충돌 / 충돌 제약 ON 은 안전 여유 약 0.6 m 를 지킴)

- **weight 1, 10**: 부딪힌 계획의 총 J 가 안전 계획보다 **더 쌉니다.** solver 가 길을 못 찾은 게 아니라, 비용 함수가 "벌금 내고 부딪히는 쪽"을 더 좋은 답으로 매긴 것입니다 — 섹션 4 의 주장 그대로입니다.
- **weight 100**: 채점판은 4/4 지만 clear 는 +0.28 m 입니다. 안전 여유 0.6 m 중 0.3 m 를 벌금 내고 침범하는 쪽이 여전히 쌉니다.
- weight 를 더 키우면 (1000 → clear +0.54 m) 안전 여유에 다가가지만, **얼마나 키워야 충분한지 미리 알 수 없습니다.** 아래 `avoid` 처럼 아무리 키워도 안 되는 경우도 있습니다. 제약은 weight 튜닝 없이 바로 지킵니다.

> **결론**: 벌금(cost)은 흥정, 제약(constraint)은 금지 → **안전은 제약으로**.
> (Step 03 에서 비용이 더 낮은 후보도 충돌이면 버린 것이 같은 원리입니다.)

> 생각해 보기: `avoid` 처럼 장애물이 reference **정면**에 있으면 벌금을 10000 까지 키워도 그대로 관통합니다.
> 왼쪽·오른쪽 어느 쪽으로 피해도 벌점이 똑같이 줄어서, 출발점에서 벌점이 피할 방향을 알려 주지 못하기 때문입니다.

---
# 추가 학습 — 도전 시나리오 (빠른 분들용)

기본 4가지 시나리오를 모두 통과했다면, 더 어려운 **도전 시나리오 3가지** 에 도전해 보세요.
도전 2·3 은 **일부러 실패하는 시작값**을 먼저 실행합니다. 결과판의 ✗ 를 보고 Case 1~5 처럼 원인을 하나씩 고칩니다.

| 시나리오 | 상황 | 과제 |
|---|---|---|
| `scurve` | S자 이중 차선변경 | comfort 를 키워 Tracking 이 무너지는 지점 찾기 (관찰) |
| `slalom` | 정지차 2대를 지그재그로 회피 | 시작값의 ✗ 고치기: 제약 1개 + weight 1개 |
| `wall` | L1·L2 두 차선이 동시에 막힘 | 시작값의 ✗ 고치기: 제약 2개 |


In [ ]:
# 도전 시나리오의 reference path 와 장애물 배치 미리 보기
PlotReferencePathPreview(['scurve', 'slalom', 'wall'])

## 도전 1 — S자 이중 차선변경 (`scurve`)

reference 가 L1→L3 로 올라갔다가 L1 로 되돌아오는 **S자** 경로입니다.
이 시나리오는 웬만한 weight 로 4/4 를 통과합니다 (track 1~50, comfort 0~1000 모두 통과).
그렇다면 `comfort` 를 얼마나 키워야 S자를 다 돌지 못하게 될까요? **'부드러움의 대가'가 나타나는 자릿수**를 찾아봅니다.

In [ ]:
##### TODO 20 #####
# comfort_big 정하기: 1e3 -> 1e4 -> 1e5 ... 처럼 10배씩 키워 가며 다시 실행, Tracking 이 ✗ 가 되는 값 찾기
#   - Tracking ✗ 기준: 채점판 RMSE > 1.0 m
#   - 기록: ✗ 가 처음 나온 값, 그때의 RMSE 와 max|dsteer|
#################

comfort_big = ...

#################
assert isinstance(comfort_big, (int, float)), 'comfort_big 미입력: 숫자로 채우기 (예: 1e3)'

W_challenge1 = {'track': 10, 'heading': 10, 'comfort': comfort_big}
res_scurve = Run(W_challenge1, scenario='scurve', title=f'도전 1: comfort = {comfort_big:g}')

# 생각해 보기: 종합 미션의 comfort 20 보다 몇 배에서 Tracking 이 무너졌나요?
# δ̇ 는 값 자체가 작아서 (이 시나리오 최대 0.16 rad/s, 제곱하면 0.03 이하) 벌점이 의미 있으려면 weight 가 아주 커야 합니다.

## 도전 2 — 정지차 2대 슬라럼 (`slalom`)

reference 는 L1 직진이지만, **L1 에 한 대, L2 에 한 대** 가 엇갈려 서 있습니다.
첫 차를 피해 올라가면 둘째 차가 막고 있어서, **올라갔다 내려오는 지그재그**가 강제됩니다.

먼저 아래 **시작값**을 그대로 실행해 어떤 항목이 ✗ 인지 확인합니다.

In [ ]:
# 도전 2 시작값: 일부러 실패하는 설정 (그대로 실행 -> ✗ 항목 확인)
W_challenge2 = {'track': 10, 'heading': 10, 'comfort': 1}
use_collision = False
res_slalom = Run(W_challenge2, scenario='slalom', use_collision=use_collision,
                 title='도전 2 시작값: 무엇이 ✗ 인가?')

In [ ]:
##### TODO 21 #####
# 시작값의 ✗ 를 하나씩 고치기: W_challenge2 와 use_collision 을 다시 정하기 (시작값을 복사해 원인만 수정)
#   1) ✗ 항목이 cost(weight) 문제인지 constraint 문제인지 판단 -> 하나만 고쳐 실행
#   2) 고친 뒤 새로 ✗ 가 생기면 그 항목도 같은 방법으로
#   (hint) 지그재그 회피를 만드는 것은 constraint, 그 지그재그를 부드럽게 만드는 것은 weight
#################

W_challenge2 = {
    'track':    ...,
    'heading':  ...,
    'comfort':  ...,
}
use_collision = None

#################
assert use_collision in (True, False), 'use_collision 미입력: True 또는 False 로 채우기'

res_slalom = Run(W_challenge2, scenario='slalom', use_collision=use_collision,
                 title='도전 2: 정지차 2대 슬라럼')

## 도전 3 — 두 차선 동시 봉쇄 (`wall`)

L1(reference 차선)과 L2 가 **동시에** 막혀 있습니다. 좋은 planner 라면 reference 를 잠시 포기하고
**반대쪽 L0 로 크게 우회**해야 합니다. 실패 case ④ 에서 본 "soft 목표보다 hard 안전이 우선"의 종합판입니다.

이번 시작값은 weight 는 종합 미션과 같고 **제약 2개가 모두 꺼져** 있습니다. 먼저 그대로 실행해 봅니다.

In [ ]:
# 도전 3 시작값: weight 는 종합 미션과 같고, 제약 2개가 모두 꺼져 있음 (그대로 실행 -> ✗ 항목 확인)
W_challenge3 = {'track': 10, 'heading': 10, 'comfort': 20}
use_collision = False
use_road = False
res_wall = Run(W_challenge3, scenario='wall', use_collision=use_collision, use_road=use_road,
               title='도전 3 시작값: 무엇이 ✗ 인가?')

In [ ]:
##### TODO 22 #####
# 제약을 하나씩 켜 가며 4/4 초록 만들기 (W_challenge3 는 그대로)
#   1) 먼저 충돌 제약만 켜고 실행 -> 어떤 항목이 새로 ✗ 가 되는지 확인
#   2) 그 원인을 막는 제약을 마저 켜기
#################

use_collision = None
use_road = None

#################
assert use_collision in (True, False), 'use_collision 미입력: True 또는 False 로 채우기'
assert use_road in (True, False), 'use_road 미입력: True 또는 False 로 채우기'

res_wall = Run(W_challenge3, scenario='wall',
               use_collision=use_collision, use_road=use_road,
               title='도전 3: 두 차선 동시 봉쇄')

# [추가 실험] Tracking 이 초록(✓)인 이유를 생각해 보세요.
# 채점은 장애물 주변 ±12 m 를 빼고 계산합니다 — '회피 중 이탈'은 허용하고, 그 밖에서는 reference 복귀를 요구하는 것입니다.

In [ ]:
# (자유 실험 공간)
# 사용 가능한 시나리오: 'curve', 'avoid', 'badref', 'mission', 'scurve', 'slalom', 'wall'
# 예: res = Run({'track':8, 'heading':12, 'comfort':1.5}, scenario='slalom', title='slalom challenge')
# 예: ShowWeightSweep('comfort', [1, 10, 100], base_weights={'track':10,'heading':10,'comfort':20}, scenario='scurve')


---
## 정리

- **Bicycle model**은 `KinematicBicycleStep()`과 `Simulate()`에서 trajectory를 만듭니다.
- **Cost (soft)** = 추종(`track`, `heading`) + 승차감(`comfort`). weight로 planner의 성격을 정합니다.
- **Constraint (hard)** = 충돌·도로·제어 한계. 무슨 일이 있어도 지켜야 하는 안전선입니다.
- 충돌을 **벌금(cost)** 으로만 막으면, 벌금이 싸면 부딪히고 비싸도 안전 여유를 정확히 지키지 못합니다 (심화). 그래서 안전은 제약으로 둡니다.
- `Run()` = `Optimize()`(solver 실행) + 결과 그림·채점판입니다. 최적화 계산은 여러분이 완성한 model · cost · constraint 가 담당합니다.
- 이 구조가 그대로 **MPC**로 확장되어 매 순간 다시 풀리며 실제 차를 움직입니다.
